In [1]:
!pip install requests

In [2]:
import requests

from google.colab import userdata
API_KEY = userdata.get("LAST_FMKEY")   # the ONLY place the key goes

def get_similar(artist, track):
    url = "http://ws.audioscrobbler.com/2.0/"
    params = {
        "method": "track.getsimilar",
        "artist": artist,
        "track": track,
        "api_key": API_KEY,
        "format": "json",
        "limit": 10,
    }
    data = requests.get(url, params=params).json()

    # if Last.fm sent back an error, show it instead of crashing
    if "error" in data:
        print("Last.fm error:", data["message"])
        return []

    return data["similartracks"]["track"]

songs = get_similar("Arijit Singh", "Tum Hi Ho")
for s in songs:
    print(s["artist"]["name"], "-", s["name"], "| match:", s["match"])

KK - Dil Ibaadat | match: 1.0
Gajendra Verma - Mann Mera | match: 0.971174
Arijit Singh - Tu Hi Hai Aashiqui | match: 0.959554
Roop Kumar Rathod - Maula Mere Maula | match: 0.737121
Mohit Chauhan - Tune Jo Na Kaha | match: 0.638513
Mohit Chauhan - Masakali | match: 0.621832
Sukhwinder Singh - Haule Haule | match: 0.609549
Atif Aslam - Tere Liye | match: 0.602583
Arijit Singh - Milne Hai Mujhse Aayi | match: 0.516104
K.K. - Dil Kyun Yeh Mera | match: 0.374278


In [3]:
from collections import defaultdict

def build_pool(artist, track, depth=2, per_song=10):
    scores = defaultdict(float)   # (artist, track) -> total score
    frontier = [(artist, track, 1.0)]

    for level in range(depth):
        next_frontier = []
        for a, t, weight in frontier:
            for s in get_similar(a, t)[:per_song]:
                key = (s["artist"]["name"], s["name"])
                w = weight * float(s["match"])   # score shrinks the further you go
                scores[key] += w
                next_frontier.append((key[0], key[1], w))
        # only expand the strongest songs, to keep API calls low
        next_frontier.sort(key=lambda x: x[2], reverse=True)
        frontier = next_frontier[:5]

    scores.pop((artist, track), None)   # remove the seed song itself
    return scores

pool = build_pool("Arijit Singh", "Tum Hi Ho")

ranked = sorted(pool.items(), key=lambda x: x[1], reverse=True)
for (a, t), sc in ranked[:20]:
    print(f"{a} - {t} | {sc:.3f}")

KK - Dil Ibaadat | 3.877
Gajendra Verma - Mann Mera | 3.249
Mohit Chauhan - Tune Jo Na Kaha | 2.961
Sukhwinder Singh - Haule Haule | 2.775
Arijit Singh - Tu Hi Hai Aashiqui | 2.766
Mohit Chauhan - Masakali | 1.769
Roop Kumar Rathod - Maula Mere Maula | 1.417
K.K. - Abhi Abhi | 1.234
Mithoon - Aye Khuda | 1.212
Atif Aslam - Tere Liye | 0.974
Kishore Kumar - Pal Pal Dil Ke Paas (From "Blackmail") | 0.956
Pritam - Pehli Nazar Mein | 0.924
K.K. - Dil Kyun Yeh Mera | 0.805
Javed Ali - Deewana Kar Raha Hai | 0.734
Lata Mangeshkar - Ham Tere Pyar Mein | 0.716
Mohit Chauhan - Abhi Kuch Dino Se | 0.639
Rahat Fateh Ali Khan - Isq Risk | 0.603
Arijit Singh - Phir Le Aya Dil | 0.595
Sonu Nigam - Apnaa Mujhe Tu Lagaa | 0.591
Arijit Singh - Uska Hi Banana | 0.531


In [4]:
import json, os, re, random

HISTORY_FILE = "history.json"

def load_history():
    if os.path.exists(HISTORY_FILE):
        return json.load(open(HISTORY_FILE))
    return {}

def save_history(h):
    json.dump(h, open(HISTORY_FILE, "w"))

def norm_artist(name):
    # "K.K." and "KK" both become "kk"
    return re.sub(r"[^a-z0-9]", "", name.lower())

def record(artist, track, event):
    """event: 'finish', 'replay' or 'skip'"""
    h = load_history()
    key = f"{norm_artist(artist)}|{track.lower()}"
    h[key] = h.get(key, 0) + {"finish": 1, "replay": 2, "skip": -3}[event]
    save_history(h)

def make_queue(pool, size=15):
    h = load_history()
    scored = []
    for (a, t), sc in pool.items():
        key = f"{norm_artist(a)}|{t.lower()}"
        sc += h.get(key, 0) * 0.5             # your taste
        if key not in h:
            sc += random.uniform(0, 0.4)      # discovery bonus
        scored.append([sc, a, t])

    scored.sort(reverse=True)
    queue, recent = [], []
    while scored and len(queue) < size:
        pick = None
        for item in scored:                   # best song whose artist isn't recent
            if norm_artist(item[1]) not in recent:
                pick = item
                break
        pick = pick or scored[0]
        scored.remove(pick)
        queue.append((pick[1], pick[2]))
        recent = (recent + [norm_artist(pick[1])])[-2:]   # last 2 artists
    return queue

# try it: pretend you skipped one and replayed another
record("KK", "Dil Ibaadat", "skip")
record("Mohit Chauhan", "Masakali", "replay")

for i, (a, t) in enumerate(make_queue(pool), 1):
    print(i, a, "-", t)

1 Gajendra Verma - Mann Mera
2 Sukhwinder Singh - Haule Haule
3 Mohit Chauhan - Tune Jo Na Kaha
4 Arijit Singh - Tu Hi Hai Aashiqui
5 KK - Dil Ibaadat
6 Mohit Chauhan - Masakali
7 Mithoon - Aye Khuda
8 Roop Kumar Rathod - Maula Mere Maula
9 K.K. - Abhi Abhi
10 Kishore Kumar - Pal Pal Dil Ke Paas (From "Blackmail")
11 Pritam - Pehli Nazar Mein
12 Lata Mangeshkar - Ham Tere Pyar Mein
13 Javed Ali - Deewana Kar Raha Hai
14 Atif Aslam - Tere Liye
15 K.K. - Dil Kyun Yeh Mera


In [5]:
from google.colab import drive
import os

try:
    print("Attempting to mount Google Drive...")
    drive.mount('/content/drive')
    HISTORY_FILE = "/content/drive/MyDrive/music_history.json"
    print("Successfully mounted Google Drive! History will be saved to your Drive.")
except Exception as e:
    print(f"Could not mount Google Drive ({type(e).__name__}).")
    print("Falling back to local runtime storage (history.json).")
    HISTORY_FILE = "history.json"

Attempting to mount Google Drive...
Mounted at /content/drive
Successfully mounted Google Drive! History will be saved to your Drive.


In [6]:
def make_queue_v2(pool, size=15):
    h = load_history()
    # drop songs you've skipped twice or more (score <= -6)
    pool = {k: v for k, v in pool.items()
            if h.get(f"{norm_artist(k[0])}|{k[1].lower()}", 0) > -6}
    return make_queue(pool, size)

def session(pool):
    queue = make_queue_v2(pool)
    for a, t in queue:
        print(f"\nNow playing: {a} - {t}")
        r = input("[f]inished  [r]eplayed  [s]kipped  [q]uit: ").strip().lower()
        if r == "q":
            break
        if r in ("f", "r", "s"):
            record(a, t, {"f": "finish", "r": "replay", "s": "skip"}[r])
    print("\nSaved. Run session(pool) again to see how the queue changed.")

session(pool)


Now playing: KK - Dil Ibaadat
[f]inished  [r]eplayed  [s]kipped  [q]uit: s

Now playing: Gajendra Verma - Mann Mera
[f]inished  [r]eplayed  [s]kipped  [q]uit: f

Now playing: Mohit Chauhan - Tune Jo Na Kaha
[f]inished  [r]eplayed  [s]kipped  [q]uit: q

Saved. Run session(pool) again to see how the queue changed.


In [7]:
songs = get_similar("Metric", "Who Would You Be for Me")
for s in songs:
    print(s["artist"]["name"], "-", s["name"], "| match:", s["match"])

In [8]:
pool = build_pool("Metric", "Who Would You Be for Me")
for i, (a, t) in enumerate(make_queue_v2(pool), 1):
    print(i, a, "-", t)

In [9]:
def search_track(query, limit=5):
    params = {
        "method": "track.search",
        "track": query,
        "api_key": API_KEY,
        "format": "json",
        "limit": limit,
    }
    data = requests.get("http://ws.audioscrobbler.com/2.0/", params=params).json()
    return data["results"]["trackmatches"]["track"]

for r in search_track("Who Would You Be for Me"):
    print(r["artist"], "-", r["name"])

Metric - Who Would You Be for Me
metricmusic - Who Would You Be For Me
The Libertines - The Man Who Would Be King
dodie - Would You Be So Kind
Streetlight Manifesto - Would You Be Impressed?


In [10]:
def build_multi_pool(seeds, depth=2):
    combined = defaultdict(float)

    for artist, track in seeds:
        pool = build_pool(artist, track, depth)
        if not pool:
            print(f"No data for {artist} - {track}, skipping")
            continue

        # normalize so every seed has equal influence (best song = 1.0)
        top = max(pool.values())
        for key, sc in pool.items():
            combined[key] += sc / top

    # remove the seeds themselves from the results
    for seed in seeds:
        combined.pop(seed, None)

    return combined

my_seeds = [
    ("Arijit Singh", "Tum Hi Ho"),
    ("Metric", "Who Would You Be for Me"),
    # add 3-5 more of your real favorites here
]

pool = build_multi_pool(my_seeds)

for i, (a, t) in enumerate(make_queue_v2(pool, size=20), 1):
    print(i, a, "-", t)

No data for Metric - Who Would You Be for Me, skipping
1 Gajendra Verma - Mann Mera
2 Mohit Chauhan - Tune Jo Na Kaha
3 Sukhwinder Singh - Haule Haule
4 Arijit Singh - Tu Hi Hai Aashiqui
5 Mohit Chauhan - Masakali
6 Kishore Kumar - Pal Pal Dil Ke Paas (From "Blackmail")
7 K.K. - Abhi Abhi
8 Atif Aslam - Tere Liye
9 Roop Kumar Rathod - Maula Mere Maula
10 Lata Mangeshkar - Ham Tere Pyar Mein
11 Rahat Fateh Ali Khan - Isq Risk
12 Pritam - Pehli Nazar Mein
13 Atif Aslam - Tere Bin
14 Arijit Singh - Aasan Nahin Yahan
15 Sonu Nigam - Apnaa Mujhe Tu Lagaa
16 KK - Hai Junoon
17 Mithoon - Aye Khuda
18 Asha Bhosle, Mohammed Rafi - Chura Liya Hai Tumne Jo Dil Ko
19 Javed Ali - Deewana Kar Raha Hai
20 Mohit Chauhan - Abhi Kuch Dino Se


In [11]:
def check_seeds(candidates):
    good = []
    for artist, track in candidates:
        n = len(get_similar(artist, track))
        print(("OK  " if n else "NONE"), artist, "-", track, f"({n} similar)")
        if n:
            good.append((artist, track))
    return good

candidates = [
    ("Metric", "Help I'm Alive"),
    ("Metric", "Gimme Sympathy"),
    ("Metric", "Black Sheep"),
    # add other English/non-Hindi songs you actually like
]

good = check_seeds(candidates)

OK   Metric - Help I'm Alive (10 similar)
OK   Metric - Gimme Sympathy (10 similar)
OK   Metric - Black Sheep (10 similar)


In [12]:
my_seeds = [("Arijit Singh", "Tum Hi Ho")] + good
pool = build_multi_pool(my_seeds)

for i, (a, t) in enumerate(make_queue_v2(pool, size=20), 1):
    print(i, a, "-", t)

1 Metric - Sick Muse
2 Gajendra Verma - Mann Mera
3 Yeah Yeah Yeahs - Maps
4 Metric - Collect Call
5 Mohit Chauhan - Tune Jo Na Kaha
6 The Killers - Somebody Told Me
7 Metric - Gold Guns Girls
8 Stars - Elevator Love Letter
9 The Killers - Mr. Brightside
10 Metric - Satellite Mind
11 Sukhwinder Singh - Haule Haule
12 Mother Mother - Hayloft
13 Metric - Combat Baby
14 Stars - Ageless Beauty
15 Sex Bob-Omb - We Are Sex Bob-Omb
16 Metric - Blindness
17 Broken Social Scene - 7/4 (Shoreline)
18 Franz Ferdinand - Take Me Out
19 Metric - Front Row
20 Mother Mother - Problems


In [13]:
def make_queue_v3(pool, size=20, max_per_artist=2):
    h = load_history()
    scored = []
    for (a, t), sc in pool.items():
        key = f"{norm_artist(a)}|{t.lower()}"
        hv = h.get(key, 0)
        if hv <= -6:                      # skipped twice: gone
            continue
        sc += hv * 0.5
        if key not in h:
            sc += random.uniform(0, 0.4)  # discovery bonus
        scored.append([sc, a, t])

    scored.sort(reverse=True)
    queue, recent, count = [], [], defaultdict(int)
    while scored and len(queue) < size:
        pick = None
        for item in scored:
            na = norm_artist(item[1])
            if na not in recent and count[na] < max_per_artist:
                pick = item
                break
        if pick is None:
            break
        scored.remove(pick)
        na = norm_artist(pick[1])
        count[na] += 1
        queue.append((pick[1], pick[2]))
        recent = (recent + [na])[-2:]
    return queue


_pool_cache = {}

def next_song(seeds, played_recently=()):
    k = tuple(seeds)
    if k not in _pool_cache:              # only hits Last.fm once per seed set
        _pool_cache[k] = build_multi_pool(seeds)
    for a, t in make_queue_v3(_pool_cache[k], size=30):
        if (a, t) not in played_recently:
            return (a, t)

# test it
my_seeds = [
    ("Arijit Singh", "Tum Hi Ho"),
    ("Metric", "Help I'm Alive"),
    # add 2-3 more Hindi seeds you like here
]
played = []
for _ in range(5):
    song = next_song(my_seeds, played)
    played.append(song)
    print(song)

('Gajendra Verma', 'Mann Mera')
('Metric', 'Front Row')
('Metric', 'Satellite Mind')
('Arijit Singh', 'Tu Hi Hai Aashiqui')
('Metric', 'Gold Guns Girls')


In [14]:
def next_song(seeds, played_recently=(), max_per_artist=2, window=10):
    k = tuple(seeds)
    if k not in _pool_cache:
        _pool_cache[k] = build_multi_pool(seeds)

    recent = list(played_recently)
    last_two = [norm_artist(a) for a, _ in recent[-2:]]
    window_artists = [norm_artist(a) for a, _ in recent[-window:]]

    # the cap is enforced below, so turn it off inside the queue builder
    candidates = make_queue_v3(_pool_cache[k], size=50, max_per_artist=50)

    for a, t in candidates:
        na = norm_artist(a)
        if (a, t) in recent:
            continue                                  # already played
        if na in last_two:
            continue                                  # same artist too soon
        if window_artists.count(na) >= max_per_artist:
            continue                                  # artist used up in last 10 songs
        return (a, t)

    # fallback: everything was filtered, so just avoid exact repeats
    for a, t in candidates:
        if (a, t) not in recent:
            return (a, t)

played = []
for _ in range(10):
    song = next_song(my_seeds, played)
    played.append(song)
    print(song)

('Gajendra Verma', 'Mann Mera')
('Metric', 'Satellite Mind')
('Sukhwinder Singh', 'Haule Haule')
('Arijit Singh', 'Tu Hi Hai Aashiqui')
('Metric', 'Gold Guns Girls')
('Mohit Chauhan', 'Tune Jo Na Kaha')
('The Naked and Famous', 'Young Blood')
('Roop Kumar Rathod', 'Maula Mere Maula')
('Yeah Yeah Yeahs', 'Soft Shock')
('Mohit Chauhan', 'Masakali')


In [15]:
print(len(get_similar("Arijit Singh", "Tum Hi Ho")))

10
